> ⚠️ **Nota de Reproducibilidad**
>
> Este cuaderno documenta el proceso histórico de etiquetado del dataset.
> **No es necesario ejecutarlo** para reproducir el entrenamiento de los modelos del PDG.
> El dataset ya etiquetado está disponible en `data/data_cleaned.xlsx`.
>
> Para ejecutarlo se requieren los pesos locales del clasificador RAD-ALERT
> (archivos `config.json`, `tokenizer` y `model.safetensors`) y acceso a la variable
> de entorno `MODELS_DIR` configurada en un archivo `.env` local.

---

# Inferencia RAD-ALERT — Etiquetado del Dataset

Este notebook toma el dataset limpio (`data_cleaned.xlsx`), normaliza el texto de los informes radiológicos
y aplica el modelo RAD-ALERT para clasificar cada registro como **Crítico** o **No Crítico**.
El resultado es el dataset etiquetado que alimenta todos los notebooks de entrenamiento del PDG.


In [6]:
# Dependencias adicionales requeridas para ejecutar la inferencia con RAD-ALERT.
# Si ya están instaladas en tu entorno, puedes saltar esta celda.
# %pip install --upgrade --no-cache-dir "tokenizers==0.20.3" "transformers==4.46.3" Unidecode


In [7]:
# ─── Librerías ─────────────────────────────────────────────────
import pandas as pd
import re
import warnings
from pathlib import Path
from unidecode import unidecode
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# ─── Variables de entorno ────────────────────────────────────────
from dotenv import load_dotenv
import os

load_dotenv()
DATA_DIR   = os.getenv('DATA_DIR',   '../data')
MODELS_DIR = os.getenv('MODELS_DIR', '../models/rad-alert')

warnings.filterwarnings('ignore')
print(f'DATA_DIR   → {DATA_DIR}')
print(f'MODELS_DIR → {MODELS_DIR}')


DATA_DIR   → ../data
MODELS_DIR → ../models/rad-alert


### 1. Carga del dataset


In [8]:
# Cargar el dataset limpio generado en el notebook 0.0
df = pd.read_excel(f'{DATA_DIR}/data_cleaned.xlsx')
print(f'Registros cargados: {len(df)}')
df.head(2)


Registros cargados: 3977


,Fecha(dd/mm/yyyy),Modalidad,Estudio,Técnica,Datos Clínicos,Hallazgos,Opinión,Reporte estructurado,Estudio Complementario Sugerido,Hallazgo Crítico,Estudio Normal
0,2019-01-01 00:00:00,CT,tomografia computada de craneo simple,"en equipo multidetector, se realizan cortes ax...",cefalea intensa con signos de alarma. descarta...,área de hipodensidad en la sustancia blanca su...,hipodensidad descrita en el lóbulo frontal izq...,0,1,0,0
1,2019-01-01 00:00:00,CT,tomografia computada de craneo simple,se realizan cortes axiales desde fosa posterio...,tce. cefalea y emesis persistente.,signos de sangrado. hematoma subdural: no. hem...,estudio sin evidencia de lesiones traumáticas ...,1,0,0,0


### 2. Preparación del input

El texto de `Hallazgos` y `Opinión` ya fue normalizado en el notebook `0.0`.
Aquí simplemente se concatenan ambas columnas para construir el input del modelo.


In [9]:
# El tokenizador de RAD-ALERT maneja su propia normalización interna.
# Solo concatenamos las columnas de texto ya limpias desde data_cleaned.xlsx.
df['texto_input'] = df['Hallazgos'].fillna('') + ' ' + df['Opinión'].fillna('')
print(f'Input preparado para {len(df)} registros.')

# Mostrar un ejemplo del texto concatenado resultante
print('\nEjemplo del primer registro concatenado:')
print(df['texto_input'].iloc[0][:300] + '...')


Input preparado para 3977 registros.


### 3. Carga del modelo RAD-ALERT

Se carga el clasificador desde la ruta local definida en `MODELS_DIR`.
Si el modelo no está disponible, esta celda lanzará un error claro.


In [10]:
model_path = Path(MODELS_DIR)
print(f'Ruta del modelo: {model_path}')
print(f'Existe: {model_path.exists()}')

try:
    if not model_path.exists():
        raise FileNotFoundError(f'No existe la ruta del modelo: {model_path}')

    tokenizer = AutoTokenizer.from_pretrained(str(model_path), local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path), local_files_only=True)

    device = torch.device('cpu')
    model.to(device)
    model.eval()
    print(f'Modelo cargado correctamente en {device}.')
except Exception as e:
    print(f'Error al cargar el modelo: {e}')
    print('Verifica MODELS_DIR en tu .env y que la carpeta tenga config.json y model.safetensors.')


Ruta del modelo: ..\models\rad-alert
Existe: False
Error al cargar el modelo: No existe la ruta del modelo: ..\models\rad-alert
Verifica MODELS_DIR en tu .env y que la carpeta tenga config.json y model.safetensors.


### 4. Inferencia sobre el dataset


In [ ]:
# Función para predecir
def predict_critical(text):
    if not text.strip():
        return "No crítico", 0.0
        
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        # Ajusta el índice (1 o 0) según qué clase represente 'Crítico' en tu modelo
        prob_critico = probs[0][1].item() 
        
    pred = "Crítico" if prob_critico >= 0.5 else "No crítico"
    return pred, prob_critico

# Aplicar a todo el dataset
# Descomenta las siguientes líneas cuando el modelo esté configurado:
print("Iniciando inferencia...")
df['rad_prediccion'], df['rad_score'] = zip(*df['texto_normalizado'].apply(predict_critical))
print(df['rad_prediccion'].value_counts())


### 5. Filtrado y exportación


In [11]:
# Filtrar y Exportar
# Descomenta esto después de ejecutar la inferencia exitosamente.

df_criticos = df[df['rad_prediccion'] == 'Crítico'].copy()
print(f"Total de registros Críticos identificados por RAD-ALERT: {len(df_criticos)}")

# Guardar el dataset final como rad_criticos.xlsx
df_criticos.to_excel('../data/rad_criticos.xlsx', index=False)
print("Exportado exitosamente a '../data/rad_criticos.xlsx'")


KeyError: 'rad_prediccion'

### 6. Análisis visual de los textos clasificados


In [12]:
# Nubes de palabras por clase usando columnas: texto_normalizado y rad_prediccion
from wordcloud import WordCloud

# Cambia esta ruta por tu archivo Excel
ruta_excel = '../data/rad_criticos.xlsx'

df_wc = pd.read_excel(ruta_excel)

# Validar columnas requeridas
columnas_requeridas = {'texto_normalizado', 'rad_prediccion'}
faltantes = columnas_requeridas - set(df_wc.columns)
if faltantes:
    raise ValueError(f"Faltan columnas en el Excel: {faltantes}")

# Asegurar tipos
df_wc['texto_normalizado'] = df_wc['texto_normalizado'].fillna('').astype(str)
df_wc['rad_prediccion'] = df_wc['rad_prediccion'].fillna('').astype(str).str.strip()

# Separar textos por clase
texto_critico = ' '.join(df_wc.loc[df_wc['rad_prediccion'].str.lower() == 'crítico', 'texto_normalizado'])
texto_no_critico = ' '.join(df_wc.loc[df_wc['rad_prediccion'].str.lower() == 'no crítico', 'texto_normalizado'])

# Evitar error si una clase no tiene texto
if not texto_critico.strip():
    texto_critico = 'sin_datos'
if not texto_no_critico.strip():
    texto_no_critico = 'sin_datos'

# Crear nubes
wc_critico = WordCloud(width=1200, height=700, background_color='white', colormap='Reds').generate(texto_critico)
wc_no_critico = WordCloud(width=1200, height=700, background_color='white', colormap='Blues').generate(texto_no_critico)

# Mostrar ambas en una figura
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(wc_critico, interpolation='bilinear')
axes[0].set_title('Nube de palabras - Crítico', fontsize=14)
axes[0].axis('off')

axes[1].imshow(wc_no_critico, interpolation='bilinear')
axes[1].set_title('Nube de palabras - No crítico', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined